# inference-mode-step — worked example 2: Equivalent step with a no_grad block

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inference-mode-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Wrapping the step body in `with t.no_grad():` achieves the same permission as the decorator: autograd is disabled for the scope, so the in-place leaf update is legal. For this update pattern the two forms produce identical parameter trajectories.

## Worked solution

We write the same optimizer with the context-manager form instead of the decorator.

1. `step` has no decorator. Instead its entire body is inside `with t.no_grad():`.
2. Within the block, the bare in-place update `p -= self.lr * p.grad` runs for each param with a gradient. The `no_grad` scope is what makes the leaf mutation legal.
3. We run this optimizer and a decorator-style one side by side on the same starting weight and identical gradients for several steps.
4. Because both merely disable grad tracking around the same arithmetic, the parameter trajectories match exactly — we assert `allclose` to confirm the equivalence.

In [ ]:
import torch as t

t.manual_seed(1)

class ContextSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr
    def step(self):
        with t.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad
    def zero_grad(self):
        for p in self.params:
            p.grad = None

class DecoratedSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr
    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad
    def zero_grad(self):
        for p in self.params:
            p.grad = None

a = t.tensor([3.0], requires_grad=True)
b = a.detach().clone().requires_grad_(True)
oa, ob = ContextSGD([a], 0.1), DecoratedSGD([b], 0.1)
for _ in range(5):
    oa.zero_grad(); ob.zero_grad()
    ((a - 1) ** 2).backward(); ((b - 1) ** 2).backward()
    oa.step(); ob.step()
print('trajectories match:', bool(t.allclose(a.detach(), b.detach(), atol=1e-6)))